# CT Corruption Detection - Cropping & Blurring Augmentation + Architecture Benchmarking

**Purpose:** Benchmark cropping and blurring corruption types across 3 architectures (ResNet-18, U-Net, EfficientNet).
Preprocessing code reused from Gnanavel's ct_corruption_medicalnet.ipynb for fair comparison.

### New Corruption Types:
- **Anatomical Cropping** - 5 sub-modes (top, bottom, both, center, random) - simulates incomplete scan coverage
- **Gaussian Blur** - motion artifacts / focal spot degradation
- **Defocus Blur** - focal plane shift / collimation errors

### Combo Modes: single (1 type), double (2 types), triple (all 3)

### Architectures Benchmarked:
- **3D ResNet-18** (MedicalNet pretrained)
- **3D U-Net** (MONAI)
- **3D EfficientNet** (MONAI)

## 1) Imports & Setup

In [ ]:
import time
NOTEBOOK_START = time.time()

import os, glob, math, random, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import SimpleITK as sitk
from scipy.ndimage import zoom, gaussian_filter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_auc_score,
    ConfusionMatrixDisplay
)

!pip -q install monai
import monai

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"MONAI:   {monai.__version__}")
print("All imports done.")

## 2) Configuration

In [ ]:
OUT_DIR = Path("./dataset_npz_crop_blur")
TARGET_SHAPE = (64, 128, 128)
HU_MIN, HU_MAX = -1000, 400

# Cropping parameters
CROP_REMOVAL_FRAC = (0.15, 0.35)
CROPPING_MODES = ("top", "bottom", "both", "center", "random")

# Blur parameters
GAUSSIAN_BLUR_SIGMA_RANGE = (1.0, 3.5)
DEFOCUS_BLUR_SIGMA_RANGE = (0.8, 2.5)

COMBO_WEIGHTS = {"single": 0.60, "double": 0.25, "triple": 0.15}

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

BATCH_SIZE = 8
PHASE1_EPOCHS = 30
PHASE2_EPOCHS = 30
PHASE1_LR = 3e-4
PHASE2_LR = 1e-5

MODEL_NAMES = ["resnet18", "unet", "efficientnet"]

BASE_DIR = "/kaggle/input/datasets/fardinabdullaacanto"
MEDICALNET_CKPT = os.path.join(BASE_DIR, "medicalnet-weights/resnet_18_23dataset.pth")

ALL_CORRUPTION_TYPES = ["anatomical_cropping", "gaussian_blur", "defocus_blur"]

print("=" * 50)
print("CONFIGURATION")
print("=" * 50)
print(f"  Output dir:          {OUT_DIR}")
print(f"  Target shape:        {TARGET_SHAPE}")
print(f"  HU window:           [{HU_MIN}, {HU_MAX}]")
print(f"  Crop removal frac:   {CROP_REMOVAL_FRAC}")
print(f"  Cropping modes:      {CROPPING_MODES}")
print(f"  Gaussian blur sigma: {GAUSSIAN_BLUR_SIGMA_RANGE}")
print(f"  Defocus blur sigma:  {DEFOCUS_BLUR_SIGMA_RANGE}")
print(f"  Combo weights:       {COMBO_WEIGHTS}")
print(f"  Data split:          {TRAIN_FRAC}/{VAL_FRAC}/{TEST_FRAC}")
print(f"  Batch size:          {BATCH_SIZE}")
print(f"  Training:            {PHASE1_EPOCHS}+{PHASE2_EPOCHS} = {PHASE1_EPOCHS+PHASE2_EPOCHS} epochs")
print(f"  Models:              {MODEL_NAMES}")
print("=" * 50)

## 3) Data Discovery - LUNA16 Subsets

In [ ]:
all_mhd_paths = []
subset_counts = {}

for i in range(10):
    candidates = [
        os.path.join(BASE_DIR, f"subset{i}", f"subset{i}"),
        os.path.join(BASE_DIR, f"subset{i}", f"subset{i}", f"subset{i}"),
        os.path.join(BASE_DIR, f"subset{i}"),
    ]

    found = False
    for path in candidates:
        mhds = sorted(glob.glob(os.path.join(path, "*.mhd")))
        if len(mhds) > 0:
            all_mhd_paths.extend(mhds)
            subset_counts[f"subset{i}"] = len(mhds)
            print(f"  subset{i}: {len(mhds):3d} scans  ({path})")
            found = True
            break

    if not found:
        mhds = sorted(glob.glob(os.path.join(BASE_DIR, f"subset{i}", "**", "*.mhd"), recursive=True))
        if len(mhds) > 0:
            all_mhd_paths.extend(mhds)
            subset_counts[f"subset{i}"] = len(mhds)
            print(f"  subset{i}: {len(mhds):3d} scans  (recursive)")
        else:
            print(f"  subset{i}:   0 scans  (NOT FOUND)")

print(f"\nTOTAL: {len(all_mhd_paths)} scans across {len(subset_counts)} subsets")

## 4) Preprocessing Functions

In [ ]:
def load_mhd(path: str) -> np.ndarray:
    """Load .mhd/.raw CT volume. Returns (D, H, W) float32 in HU."""
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    return arr.astype(np.float32)

def clip_hu(vol: np.ndarray, lo: int = -1000, hi: int = 400) -> np.ndarray:
    """Clip volume to Hounsfield Unit lung window."""
    return np.clip(vol, lo, hi)

def normalize_01(vol: np.ndarray, lo: int = -1000, hi: int = 400) -> np.ndarray:
    """Normalize clipped volume to [0, 1] range."""
    return ((vol - lo) / (hi - lo)).astype(np.float32)

def resize_dhw(vol: np.ndarray, out_shape: tuple = (64, 128, 128)) -> np.ndarray:
    """Resize volume to target (D, H, W) using trilinear interpolation."""
    D, H, W = vol.shape
    d2, h2, w2 = out_shape
    return zoom(vol, (d2 / D, h2 / H, w2 / W), order=1).astype(np.float32)

def preprocess(vol_hu: np.ndarray, out_shape: tuple = TARGET_SHAPE) -> np.ndarray:
    """Full pipeline: clip -> normalize -> resize."""
    vol = clip_hu(vol_hu, HU_MIN, HU_MAX)
    vol = normalize_01(vol, HU_MIN, HU_MAX)
    vol = resize_dhw(vol, out_shape)
    return vol

vol0 = load_mhd(all_mhd_paths[0])
print(f"Volume 0 shape: {vol0.shape}, HU range: [{vol0.min():.0f}, {vol0.max():.0f}]")

vol_mid = load_mhd(all_mhd_paths[len(all_mhd_paths) // 2])
print(f"Volume mid shape: {vol_mid.shape}, HU range: [{vol_mid.min():.0f}, {vol_mid.max():.0f}]")

vol_p = preprocess(vol0)
print(f"Preprocessed: {vol_p.shape}, range [{vol_p.min():.2f}, {vol_p.max():.2f}]")

## 5) Visualization Helpers

In [ ]:
def show_slice(vol: np.ndarray, idx: int, vmin: float = -1000, vmax: float = 400, title: str = None) -> None:
    """Display a single slice from a 3D volume."""
    plt.figure(figsize=(4, 4))
    plt.imshow(vol[idx], cmap="gray", vmin=vmin, vmax=vmax)
    plt.axis("off")
    plt.title(title or f"slice {idx}")
    plt.show()

def show_comparison(vol_a: np.ndarray, vol_b: np.ndarray, idx: int,
                    title_a: str = "A", title_b: str = "B") -> None:
    """Side-by-side comparison with absolute difference."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(vol_a[idx], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(title_a); axes[0].axis("off")
    axes[1].imshow(vol_b[idx], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(title_b); axes[1].axis("off")
    axes[2].imshow(np.abs(vol_a[idx] - vol_b[idx]), cmap="hot")
    axes[2].set_title("Absolute Difference"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

mid = vol0.shape[0] // 2
show_slice(vol0, mid, title="Original CT (lung window)")

## 6) New Corruption Functions - Cropping & Blurring

In [ ]:
def apply_anatomical_cropping(vol_hu: np.ndarray, removal_frac: float,
                              mode: str = "random",
                              rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Simulate incomplete scan coverage by cropping portions of a 3D CT volume.
  
    Five sub-modes:

    1. top: Zero out slices from the top (superior) end.
    2. bottom: Zero out slices from the bottom (inferior) end.
    3. both: Zero out slices from both ends.
    4. center: Zero out a contiguous block from the center.
    5. random: Zero out random scattered slices.

    Parameters
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    removal_frac : float
        Fraction of slices to crop, in [0, 1).
    mode : str
        One of "top", "bottom", "both", "center", "random".
    rng : np.random.Generator, optional

    """
    rng = rng or np.random.default_rng()
    D, H, W = vol_hu.shape
    n_remove = max(1, int(D * removal_frac))
    n_remove = min(n_remove, D - 1)

    result = vol_hu.copy()

    if mode == "top":
        result[:n_remove] = -1000.0

    elif mode == "bottom":
        result[D - n_remove:] = -1000.0

    elif mode == "both":
        n_top = n_remove // 2
        n_bot = n_remove - n_top
        result[:n_top] = -1000.0
        result[D - n_bot:] = -1000.0

    elif mode == "center":
        earliest_start = int(D * 0.25)
        latest_start = max(earliest_start, int(D * 0.75) - n_remove)
        start = rng.integers(earliest_start, latest_start + 1)
        result[start:start + n_remove] = -1000.0

    elif mode == "random":
        remove_indices = rng.choice(D, size=n_remove, replace=False)
        result[remove_indices] = -1000.0

    else:
        raise ValueError(f"Unknown cropping mode: {mode!r}")

    info = {
        "mode": mode,
        "slices_cropped": n_remove,
        "removal_frac_actual": round(n_remove / D, 4),
    }
    return result, info

In [ ]:
print("Testing apply_anatomical_cropping")
dummy = np.random.randn(200, 512, 512).astype(np.float32) * 200 - 500
for mode in ["top", "bottom", "both", "center", "random"]:
    result, info = apply_anatomical_cropping(dummy, removal_frac=0.25, mode=mode, rng=np.random.default_rng(42))
    assert result.shape == dummy.shape, f"{mode}: shape mismatch"
    n_air = np.sum(result == -1000.0)
    expected_air = info['slices_cropped'] * 512 * 512
    assert n_air >= expected_air * 0.99, f"{mode}: not enough air voxels"
    print(f"  {mode:10s}: shape={result.shape}, cropped {info['slices_cropped']} slices")
print("PASSED: apply_anatomical_cropping all 5 modes")

In [ ]:
def apply_gaussian_blur(vol_hu: np.ndarray, sigma: float,
                        rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Apply Gaussian blur to a 3D CT volume to simulate motion artifacts.

    This corruption models real-world scenarios where image sharpness is degraded:

    - Patient motion during acquisition (breathing, involuntary movement)
      causing blur across all three spatial dimensions.
    - Focal spot blooming in aging X-ray tubes where the electron beam
      spreads beyond the intended focal point.
    - Detector readout smearing in equipment with degraded electronics.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    sigma : float
        Standard deviation of the Gaussian kernel. Typical range: 1.0-3.5.
        Higher values produce more aggressive blur.
    rng : np.random.Generator, optional

    Returns
    -------
    tuple[np.ndarray, dict]
        (blurred_volume, info_dict) where info_dict contains:
        - "sigma": the blur kernel standard deviation used
        - "type": "gaussian"
    """
    blurred = gaussian_filter(vol_hu, sigma=sigma, mode='nearest')
    return blurred.astype(np.float32), {"sigma": float(sigma), "type": "gaussian"}

In [ ]:
print("--- Testing apply_gaussian_blur ---")
dummy = np.random.randn(100, 64, 64).astype(np.float32) * 200 - 500
blurred, info = apply_gaussian_blur(dummy, sigma=2.0, rng=np.random.default_rng(42))
print(f"  original range: [{dummy.min():.0f}, {dummy.max():.0f}]")
print(f"  blurred range:  [{blurred.min():.0f}, {blurred.max():.0f}]")
print(f"  sigma: {info['sigma']}")
assert blurred.shape == dummy.shape
assert np.abs(blurred.mean() - dummy.mean()) < 10, "Blur should preserve mean"
assert blurred.std() < dummy.std(), "Blur should reduce variance"
print("PASSED: apply_gaussian_blur")

In [ ]:
def apply_defocus_blur(vol_hu: np.ndarray, sigma: float,
                       rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Apply defocus blur to a 3D CT volume with anisotropic kernel.

    Defocus blur differs from motion blur by simulating optical degradation
    rather than kinetic effects. This corruption models:

    - Focal plane shift where the X-ray focal spot is positioned incorrectly
      relative to the detector, causing blur that varies by spatial axis.
    - Collimation errors in the beam geometry that produce anisotropic
      blur patterns (stronger blur in-plane than through-plane).
    - Detector defocus in systems where the scintillator-photodiode coupling
      has degraded, producing spatially varying blur.

    Implementation: Applies Gaussian blur with sigma scaled by [1.0, 1.5, 1.5]
    on (D, H, W) to create mild anisotropy, stronger in the H-W plane.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    sigma : float
        Base standard deviation. Typical range: 0.8-2.5.
    rng : np.random.Generator, optional

    Returns
    -------
    tuple[np.ndarray, dict]
        (blurred_volume, info_dict) where info_dict contains:
        - "sigma": base sigma value
        - "sigma_vec": anisotropic sigma vector [D, H, W]
        - "type": "defocus"
    """
    sigma_vec = (sigma, sigma * 1.5, sigma * 1.5)
    blurred = gaussian_filter(vol_hu, sigma=sigma_vec, mode='nearest')
    return blurred.astype(np.float32), {
        "sigma": float(sigma),
        "sigma_vec": [float(s) for s in sigma_vec],
        "type": "defocus"
    }

In [ ]:
print("--- Testing apply_defocus_blur ---")
dummy = np.random.randn(100, 64, 64).astype(np.float32) * 200 - 500
blurred, info = apply_defocus_blur(dummy, sigma=1.5)
print(f"  original range: [{dummy.min():.0f}, {dummy.max():.0f}]")
print(f"  blurred range:  [{blurred.min():.0f}, {blurred.max():.0f}]")
print(f"  sigma_vec: {info['sigma_vec']}")
assert blurred.shape == dummy.shape
assert np.abs(blurred.mean() - dummy.mean()) < 10
assert blurred.std() < dummy.std()
print("PASSED: apply_defocus_blur")

In [ ]:
print("=" * 60)
print("INTEGRATION TEST: All Corruption Functions")
print("=" * 60)

rng = np.random.default_rng(42)
dummy = rng.normal(-500, 200, size=(200, 512, 512)).astype(np.float32)
dummy = np.clip(dummy, -1024, 3071)
print(f"Dummy volume: shape={dummy.shape}, range=[{dummy.min():.0f}, {dummy.max():.0f}]")
print()

print("1) Anatomical Cropping:")
for mode in ["top", "bottom", "both", "center", "random"]:
    r, info = apply_anatomical_cropping(dummy, 0.25, mode=mode, rng=np.random.default_rng(42))
    assert r.shape == dummy.shape
    print(f"   {mode:10s}: shape preserved {r.shape}, cropped {info['slices_cropped']}")

print("\n2) Gaussian Blur:")
blurred, info = apply_gaussian_blur(dummy, sigma=2.0)
assert blurred.shape == dummy.shape
print(f"   shape unchanged: {blurred.shape}, sigma: {info['sigma']}")

print("\n3) Defocus Blur:")
defocused, info = apply_defocus_blur(dummy, sigma=1.5)
assert defocused.shape == dummy.shape
print(f"   shape unchanged: {defocused.shape}, sigma_vec: {info['sigma_vec']}")

print("\n4) Preprocessing after corruption:")
for name, vol in [("cropped", apply_anatomical_cropping(dummy, 0.25, "center", rng=np.random.default_rng(42))[0]),
                   ("gaussian", blurred),
                   ("defocus", defocused)]:
    pp = preprocess(vol)
    assert pp.shape == tuple(TARGET_SHAPE)
    assert pp.min() >= 0.0 and pp.max() <= 1.0
    print(f"   {name:10s}: preprocessed to {pp.shape}, range [{pp.min():.3f}, {pp.max():.3f}]")

print("\n" + "=" * 60)
print("ALL INTEGRATION TESTS PASSED")
print("=" * 60)

## 7) Combo Corruption Logic

In [ ]:
def apply_combo_corruption(vol_hu: np.ndarray, corruption_list: list[str],
                           params_dict: dict,
                           rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Apply multiple corruption types sequentially to a 3D CT volume.

    Simulates real-world scenarios where multiple quality issues occur
    simultaneously - e.g., incomplete coverage with motion blur.

    Ordering: All three corruption types preserve volume shape, so order
    is applied alphabetically for consistency.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    corruption_list : list[str]
        List of corruption type names to apply.
    params_dict : dict
        Parameters for each corruption function.
    rng : np.random.Generator, optional

    Returns
    -------
    tuple[np.ndarray, dict]
        (corrupted_volume, combined_info)
    """
    rng = rng or np.random.default_rng()

    sorted_list = sorted(corruption_list)
    combined_info: dict = {"types_applied": sorted_list}

    dispatch = {
        "anatomical_cropping": apply_anatomical_cropping,
        "gaussian_blur": apply_gaussian_blur,
        "defocus_blur": apply_defocus_blur,
    }

    for ctype in sorted_list:
        fn = dispatch[ctype]
        kwargs = params_dict.get(ctype, {})
        vol_hu, info = fn(vol_hu, **kwargs, rng=rng)
        combined_info[ctype] = info

    return vol_hu, combined_info

In [ ]:
print("--- Testing Combo Corruption ---")
rng = np.random.default_rng(42)
dummy = rng.normal(-500, 200, (200, 512, 512)).astype(np.float32)

params_2 = {
    "anatomical_cropping": {"removal_frac": 0.2, "mode": "center"},
    "gaussian_blur": {"sigma": 2.0}
}
r2, info2 = apply_combo_corruption(dummy, ["gaussian_blur", "anatomical_cropping"], params_2, rng=np.random.default_rng(42))
assert r2.shape == dummy.shape
assert len(info2["types_applied"]) == 2
print(f"  combo_2: shape preserved {r2.shape}, applied: {info2['types_applied']}")

params_3 = {
    "anatomical_cropping": {"removal_frac": 0.2, "mode": "random"},
    "gaussian_blur": {"sigma": 2.0},
    "defocus_blur": {"sigma": 1.5}
}
r3, info3 = apply_combo_corruption(dummy, ["defocus_blur", "gaussian_blur", "anatomical_cropping"], params_3, rng=np.random.default_rng(42))
assert r3.shape == dummy.shape
assert len(info3["types_applied"]) == 3
print(f"  combo_3: shape preserved {r3.shape}, applied: {info3['types_applied']}")

pp = preprocess(r3)
assert pp.shape == tuple(TARGET_SHAPE)
print(f"  combo_3 preprocessed: {pp.shape}")

print("PASSED: Combo corruption logic")

## 8) Dataset Builder

In [ ]:
def pick_scan_label(D_original: int, mode: str = "random", p_corrupt: float = 0.5,
                    rng: np.random.Generator = None) -> tuple[int, dict]:
    """
    Decide whether a scan is clean or corrupted, and generate corruption params.

    For corrupted samples, uses COMBO_WEIGHTS to decide the combo level:
    - 60% single (1 of 3 types)
    - 25% combo_2 (2 of 3 types)
    - 15% combo_3 (all 3 types)

    Parameters
    ----------
    D_original : int
        Original depth of the volume.
    mode : str
        "random", "clean", or "corrupt".
    p_corrupt : float
        Probability of corruption (default 0.5).
    rng : np.random.Generator, optional

    Returns
    -------
    tuple[int, dict]
        (label, meta) where label is 0 (clean) or 1 (corrupted).
    """
    rng = rng or np.random.default_rng()

    if mode == "clean":
        return 0, {"type": "clean"}
    if mode == "corrupt":
        is_corrupt = True
    else:
        is_corrupt = rng.random() < p_corrupt

    if not is_corrupt:
        return 0, {"type": "clean"}

    combo_keys = list(COMBO_WEIGHTS.keys())
    combo_probs = np.array([COMBO_WEIGHTS[k] for k in combo_keys])
    combo_probs = combo_probs / combo_probs.sum()
    combo_level = rng.choice(combo_keys, p=combo_probs)

    if combo_level == "single":
        n_types = 1
    elif combo_level == "double":
        n_types = 2
    else:
        n_types = 3

    chosen_types = list(rng.choice(ALL_CORRUPTION_TYPES, size=n_types, replace=False))

    params = {}
    for ctype in chosen_types:
        if ctype == "anatomical_cropping":
            frac = rng.uniform(*CROP_REMOVAL_FRAC)
            crop_mode = rng.choice(CROPPING_MODES)
            params[ctype] = {"removal_frac": float(frac), "mode": str(crop_mode)}
        elif ctype == "gaussian_blur":
            sigma = rng.uniform(*GAUSSIAN_BLUR_SIGMA_RANGE)
            params[ctype] = {"sigma": float(sigma)}
        elif ctype == "defocus_blur":
            sigma = rng.uniform(*DEFOCUS_BLUR_SIGMA_RANGE)
            params[ctype] = {"sigma": float(sigma)}

    if n_types == 1:
        ctype_label = chosen_types[0]
    elif n_types == 2:
        ctype_label = "combo_2"
    else:
        ctype_label = "combo_3"

    meta = {
        "type": ctype_label,
        "corruption_types": chosen_types,
        "params": params,
    }
    return 1, meta


def corrupt_and_preprocess(vol_hu: np.ndarray, meta: dict,
                           rng: np.random.Generator = None) -> np.ndarray:
    """
    Apply corruption (if any) then preprocess a volume to model-ready format.

    Parameters
    ----------
    vol_hu : np.ndarray
        Raw 3D CT volume in Hounsfield Units.
    meta : dict
        Metadata from pick_scan_label().
    rng : np.random.Generator, optional

    Returns
    -------
    np.ndarray
        Preprocessed volume, shape TARGET_SHAPE, range [0, 1].
    """
    rng = rng or np.random.default_rng()
    ctype = meta["type"]

    if ctype == "clean":
        return preprocess(vol_hu)

    corruption_types = meta["corruption_types"]
    params = meta["params"]

    if ctype in ("combo_2", "combo_3"):
        vol_hu, _ = apply_combo_corruption(vol_hu, corruption_types, params, rng=rng)
    else:
        single_type = corruption_types[0]
        dispatch = {
            "anatomical_cropping": apply_anatomical_cropping,
            "gaussian_blur": apply_gaussian_blur,
            "defocus_blur": apply_defocus_blur,
        }
        fn = dispatch[single_type]
        vol_hu, _ = fn(vol_hu, **params[single_type], rng=rng)

    return preprocess(vol_hu)


def make_dataset(mhd_list: list[str], split: str = "train", seed: int = 0) -> dict:
    """
    Build .npz dataset from raw .mhd scans with 50/50 clean/corrupt balance.

    Parameters
    ----------
    mhd_list : list[str]
        List of .mhd file paths.
    split : str
        "train", "val", or "test".
    seed : int
        Random seed for this split.

    Returns
    -------
    dict
        Statistics dict with counts per corruption type.
    """
    rng = np.random.default_rng(seed)
    out_path = OUT_DIR / split
    out_path.mkdir(parents=True, exist_ok=True)

    stats = {
        "clean": 0, "anatomical_cropping": 0, "gaussian_blur": 0,
        "defocus_blur": 0, "combo_2": 0, "combo_3": 0,
    }
    n_total = len(mhd_list)

    for i, mhd_path in enumerate(mhd_list):
        try:
            vol_hu = load_mhd(mhd_path)
            D_original = vol_hu.shape[0]
            label, meta = pick_scan_label(D_original, mode="random", rng=rng)
            vol_pp = corrupt_and_preprocess(vol_hu, meta, rng=rng)

            ctype = meta["type"]
            stats[ctype] = stats.get(ctype, 0) + 1

            fname = f"{split}_{i:04d}.npz"
            np.savez_compressed(
                str(out_path / fname),
                vol=vol_pp,
                label=np.int64(label),
                ctype=ctype,
            )

            if (i + 1) % 50 == 0 or (i + 1) == n_total:
                print(f"  [{split}] {i+1}/{n_total} processed")

        except Exception as e:
            print(f"  [{split}] ERROR on {mhd_path}: {e}")

    print(f"  [{split}] Done: {n_total} scans -> {out_path}")
    print(f"  [{split}] Stats: {stats}")
    return stats

In [ ]:
print("--- Testing Dataset Builder ---")
rng = np.random.default_rng(42)
stats = {"clean": 0, "anatomical_cropping": 0, "gaussian_blur": 0,
         "defocus_blur": 0, "combo_2": 0, "combo_3": 0}

for _ in range(100):
    label, meta = pick_scan_label(200, mode="random", rng=rng)
    if meta["type"] in stats:
        stats[meta["type"]] += 1

print("Distribution over 100 samples:")
for k, v in stats.items():
    print(f"  {k:25s}: {v:3d} ({v}%)")

clean_pct = stats["clean"]
corrupt_pct = 100 - clean_pct
print(f"\nClean: ~{clean_pct}%, Corrupt: ~{corrupt_pct}% (expect ~50/50)")
assert 35 <= clean_pct <= 65, f"Balance off: {clean_pct}% clean"

print("PASSED: Dataset builder distribution")

## 9) Build Dataset (3-way split)

In [ ]:
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
for split in ["train", "val", "test"]:
    (OUT_DIR / split).mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
paths = all_mhd_paths.copy()
random.shuffle(paths)

n_total = len(paths)
n_train = int(n_total * TRAIN_FRAC)
n_val = int(n_total * VAL_FRAC)
n_test = n_total - n_train - n_val

train_paths = paths[:n_train]
val_paths = paths[n_train:n_train + n_val]
test_paths = paths[n_train + n_val:]

print("=" * 50)
print("DATASET SPLIT")
print("=" * 50)
print(f"  Total scans:  {n_total}")
print(f"  Train:        {len(train_paths)} ({100*len(train_paths)/n_total:.1f}%)")
print(f"  Validation:   {len(val_paths)} ({100*len(val_paths)/n_total:.1f}%)")
print(f"  Test:         {len(test_paths)} ({100*len(test_paths)/n_total:.1f}%)")
print("=" * 50)

print("\nBuilding train set...")
train_stats = make_dataset(train_paths, split="train", seed=RANDOM_SEED)

print("\nBuilding validation set...")
val_stats = make_dataset(val_paths, split="val", seed=RANDOM_SEED + 1)

print("\nBuilding test set...")
test_stats = make_dataset(test_paths, split="test", seed=RANDOM_SEED + 2)

## 10) Dataset Statistics & Sample Visualization

In [ ]:
def label_stats(folder) -> dict:
    """Compute label and corruption-type statistics for a dataset split."""
    labels, ctypes = [], []
    for p in Path(folder).glob("*.npz"):
        d = np.load(p, allow_pickle=True)
        labels.append(int(d["label"]))
        ctypes.append(str(d["ctype"]))
    labels = np.array(labels)
    counts = {}
    for ct in ctypes:
        counts[ct] = counts.get(ct, 0) + 1
    return {"N": len(labels), "clean": int((labels == 0).sum()),
            "corrupt": int((labels == 1).sum()), "types": counts}


print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
for split in ["train", "val", "test"]:
    s = label_stats(OUT_DIR / split)
    print(f"\n  {split.upper():5s}: {s['N']} total | {s['clean']} clean | {s['corrupt']} corrupt")
    for ctype, count in sorted(s["types"].items()):
        print(f"         {ctype:25s}: {count}")
print("=" * 60)

np.random.seed(RANDOM_SEED)
train_npz_files = sorted(list((OUT_DIR / "train").glob("*.npz")))
sample_indices = np.random.choice(len(train_npz_files), size=min(8, len(train_npz_files)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Random Training Samples - Green=Clean, Red=Corrupted", fontsize=14)

for ax_idx, (ax, file_idx) in enumerate(zip(axes.flat, sample_indices)):
    data = np.load(train_npz_files[file_idx], allow_pickle=True)
    vol = data["vol"]
    label = int(data["label"])
    ctype = str(data["ctype"])

    mid = vol.shape[0] // 2
    ax.imshow(vol[mid], cmap="gray", vmin=0, vmax=1)
    color = "red" if label == 1 else "green"
    title = f"{'CORRUPT' if label == 1 else 'CLEAN'}\n{ctype}"
    ax.set_title(title, color=color, fontsize=10, fontweight="bold")
    ax.axis("off")

for ax in axes.flat[len(sample_indices):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("\n--- Dataset Build Verification ---")
for split in ["train", "val", "test"]:
    s = label_stats(OUT_DIR / split)
    assert s["N"] > 0, f"{split} is empty!"
    assert s["clean"] > 0, f"{split} has no clean samples"
    assert s["corrupt"] > 0, f"{split} has no corrupt samples"
    print(f"  {split}: {s['N']} samples, {s['types']}")
print("PASSED: Dataset build verified")

## 11) PyTorch Dataset & DataLoaders

In [ ]:
class NPZScanDataset(Dataset):
    """PyTorch Dataset that loads preprocessed .npz CT volumes."""

    def __init__(self, folder):
        self.paths = sorted(Path(folder).glob("*.npz"))
        if len(self.paths) == 0:
            raise FileNotFoundError(f"No .npz in {folder}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx], allow_pickle=True)
        vol = d["vol"].astype(np.float32)
        label = int(d["label"])
        vol = np.expand_dims(vol, axis=0)
        return torch.from_numpy(vol), torch.tensor(label, dtype=torch.float32)


train_ds = NPZScanDataset(OUT_DIR / "train")
val_ds   = NPZScanDataset(OUT_DIR / "val")
test_ds  = NPZScanDataset(OUT_DIR / "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("--- DataLoader Test ---")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
x, y = next(iter(train_loader))
print(f"Batch: x={tuple(x.shape)}, y={tuple(y.shape)}")
assert x.shape == (BATCH_SIZE, 1, 64, 128, 128) or x.shape[0] <= BATCH_SIZE
assert y.shape[0] == x.shape[0]
print("PASSED: DataLoader works")

## 12) Model Definitions - 3 Architectures

In [ ]:
!git clone https://github.com/Tencent/MedicalNet.git

import sys
sys.path.append("/kaggle/working/MedicalNet")

from models import resnet


class MedicalNetResNet18Binary(nn.Module):
    """3D ResNet-18 for binary classification using MedicalNet backbone."""

    def __init__(self) -> None:
        super().__init__()
        self.backbone = resnet.resnet18(
            sample_input_D=TARGET_SHAPE[0],
            sample_input_H=TARGET_SHAPE[1],
            sample_input_W=TARGET_SHAPE[2],
            num_seg_classes=2,
            shortcut_type='A',
            no_cuda=False
        )
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


def load_medicalnet_weights(model: nn.Module, ckpt_path: str) -> None:
    """Load MedicalNet pretrained weights into ResNet-18 backbone."""
    if not os.path.isfile(ckpt_path):
        print(f"WARNING: {ckpt_path} not found. Training from scratch.")
        return
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt.get("state_dict", ckpt)
    model_sd = model.state_dict()
    new_sd = {}
    for k, v in sd.items():
        k = k.replace("module.", "")
        if not k.startswith("backbone."):
            k = "backbone." + k
        if k in model_sd and v.shape == model_sd[k].shape:
            new_sd[k] = v
    missing, _ = model.load_state_dict(new_sd, strict=False)
    print(f"Loaded {len(new_sd)}/{len(model_sd)} tensors from MedicalNet checkpoint.")
    print(f"Missing (seg head + fc): {len(missing)} keys")


from monai.networks.nets import UNet


class UNet3DBinary(nn.Module):
    """3D U-Net for binary classification adapted from segmentation."""

    def __init__(self) -> None:
        super().__init__()
        self.backbone = UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=1,
            channels=(32, 64, 128, 256),
            strides=(2, 2, 2),
            num_res_units=2
        )
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(x)
        pooled = self.pool(feat).flatten(1)
        pooled = self.dropout(pooled)
        return self.fc(pooled).squeeze(1)


from monai.networks.nets import EfficientNetBN


class EfficientNet3DBinary(nn.Module):
    """3D EfficientNet for binary classification."""

    def __init__(self) -> None:
        super().__init__()
        self.backbone = EfficientNetBN(
            model_name="efficientnet-b0",
            spatial_dims=3,
            in_channels=1,
            num_classes=1
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x).squeeze(1)


print("All 3 model classes defined:")
print("  1. MedicalNetResNet18Binary  — MedicalNet pretrained ResNet-18")
print("  2. UNet3DBinary              — MONAI 3D U-Net")
print("  3. EfficientNet3DBinary      — MONAI 3D EfficientNet-B0")

## 13) Model Factory & Verification

In [ ]:
def create_model(name: str) -> nn.Module:
    """Factory function to create model by name."""
    if name == "resnet18":
        model = MedicalNetResNet18Binary()
        load_medicalnet_weights(model, MEDICALNET_CKPT)
        return model
    elif name == "unet":
        return UNet3DBinary()
    elif name == "efficientnet":
        return EfficientNet3DBinary()
    else:
        raise ValueError(f"Unknown model: {name}")


def count_parameters(model: nn.Module) -> int:
    """Return total number of parameters."""
    return sum(p.numel() for p in model.parameters())


def count_trainable(model: nn.Module) -> int:
    """Return number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


device = "cuda" if torch.cuda.is_available() else "cpu"
dummy_input = torch.randn(2, 1, 64, 128, 128).to(device)

print("=" * 70)
print("MODEL VERIFICATION - Forward Pass Test")
print("=" * 70)

for name in MODEL_NAMES:
    print(f"\n--- {name} ---")
    model = create_model(name).to(device)

    with torch.no_grad():
        output = model(dummy_input)

    print(f"  Input:      {tuple(dummy_input.shape)}")
    print(f"  Output:     {tuple(output.shape)}")
    print(f"  Parameters: {count_parameters(model):,}")

    assert output.shape == (2,), f"{name}: expected output (2,), got {output.shape}"
    assert torch.isfinite(output).all(), f"{name}: output contains NaN/Inf"

    probs = torch.sigmoid(output)
    assert (probs >= 0).all() and (probs <= 1).all()

    print(f"  Sigmoid:    [{probs.min().item():.4f}, {probs.max().item():.4f}]")
    print(f"  STATUS:     PASSED")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("ALL 3 MODELS VERIFIED")
print("=" * 70)

## 14) Training Functions (Model-Agnostic)

In [ ]:
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module) -> tuple[float, float]:
    """Train model for one epoch."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        n += x.size(0)
    return total_loss / max(1, n), correct / max(1, n)


@torch.no_grad()
def eval_one_epoch(model: nn.Module, loader: DataLoader,
                   criterion: nn.Module) -> tuple[float, float, np.ndarray, np.ndarray]:
    """Evaluate model for one epoch."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        total_loss += loss.item() * x.size(0)
        correct += (preds == y).sum().item()
        n += x.size(0)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_labels.extend(y.cpu().numpy().tolist())
    return total_loss / max(1, n), correct / max(1, n), np.array(all_probs), np.array(all_labels)


def freeze_early_layers_generic(model: nn.Module, model_name: str) -> None:
    """Freeze approximately the first 50% of layers."""
    if model_name == "resnet18":
        for name, param in model.named_parameters():
            if any(name.startswith(f"backbone.{l}") for l in ["conv1", "bn1", "layer1", "layer2"]):
                param.requires_grad = False
    elif model_name == "unet":
        params = list(model.parameters())
        for p in params[:len(params) // 2]:
            p.requires_grad = False
    elif model_name == "efficientnet":
        params = list(model.parameters())
        for p in params[:len(params) // 2]:
            p.requires_grad = False

    trainable = count_trainable(model)
    total = count_parameters(model)
    print(f"  [{model_name}] Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)")


def unfreeze_all(model: nn.Module) -> None:
    """Unfreeze all parameters."""
    for p in model.parameters():
        p.requires_grad = True


def train_full_pipeline(model_name: str) -> dict:
    """
    Train one model through both phases (frozen + full fine-tuning).

    Phase 1: Early layers frozen (PHASE1_EPOCHS epochs).
    Phase 2: All layers unfrozen (PHASE2_EPOCHS epochs).
    Best model saved based on validation accuracy.
    """
    print(f"\n{'=' * 75}")
    print(f"  TRAINING: {model_name}")
    print(f"{'=' * 75}")

    model = create_model(model_name).to(device)
    criterion = nn.BCEWithLogitsLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "phase": []}
    best_val_acc = 0.0
    best_epoch = 0

    freeze_early_layers_generic(model, model_name)
    opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=PHASE1_LR)
    sched1 = CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS, eta_min=1e-6)

    for ep in range(1, PHASE1_EPOCHS + 1):
        tr_l, tr_a = train_one_epoch(model, train_loader, opt1, criterion)
        va_l, va_a, _, _ = eval_one_epoch(model, val_loader, criterion)
        sched1.step()
        history["train_loss"].append(tr_l); history["train_acc"].append(tr_a)
        history["val_loss"].append(va_l); history["val_acc"].append(va_a)
        history["phase"].append(1)
        if va_a > best_val_acc:
            best_val_acc = va_a; best_epoch = ep
            torch.save(model.state_dict(), f"best_{model_name}.pth")
        if ep % 10 == 0 or ep == 1:
            print(f"  P1 {ep:02d}/{PHASE1_EPOCHS} | loss {tr_l:.4f} acc {tr_a:.3f} | val {va_l:.4f} acc {va_a:.3f}")

    unfreeze_all(model)
    opt2 = torch.optim.Adam(model.parameters(), lr=PHASE2_LR)
    sched2 = CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS, eta_min=1e-7)

    for ep in range(1, PHASE2_EPOCHS + 1):
        tr_l, tr_a = train_one_epoch(model, train_loader, opt2, criterion)
        va_l, va_a, _, _ = eval_one_epoch(model, val_loader, criterion)
        sched2.step()
        history["train_loss"].append(tr_l); history["train_acc"].append(tr_a)
        history["val_loss"].append(va_l); history["val_acc"].append(va_a)
        history["phase"].append(2)
        if va_a > best_val_acc:
            best_val_acc = va_a; best_epoch = PHASE1_EPOCHS + ep
            torch.save(model.state_dict(), f"best_{model_name}.pth")
        if ep % 10 == 0 or ep == 1:
            print(f"  P2 {ep:02d}/{PHASE2_EPOCHS} | loss {tr_l:.4f} acc {tr_a:.3f} | val {va_l:.4f} acc {va_a:.3f}")

    print(f"\n  Best val accuracy: {best_val_acc:.4f} (epoch {best_epoch})")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"history": history, "best_val_acc": best_val_acc, "best_epoch": best_epoch}


print("--- Quick Training Test (1 epoch, resnet18) ---")
_test_model = create_model("resnet18").to(device)
_test_crit = nn.BCEWithLogitsLoss()
_test_opt = torch.optim.Adam(_test_model.parameters(), lr=1e-3)
tr_l, tr_a = train_one_epoch(_test_model, train_loader, _test_opt, _test_crit)
va_l, va_a, _, _ = eval_one_epoch(_test_model, val_loader, _test_crit)
print(f"  1 epoch: train_loss={tr_l:.4f}, train_acc={tr_a:.3f}, val_acc={va_a:.3f}")
assert 0 <= tr_a <= 1 and 0 <= va_a <= 1
del _test_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("PASSED: Training loop works")

## 15) Train All Models

In [ ]:
all_results = {}

for model_name in MODEL_NAMES:
    result = train_full_pipeline(model_name)
    all_results[model_name] = result
    print(f"\n{'-' * 40}")
    print(f"  {model_name} COMPLETE - best val acc: {result['best_val_acc']:.4f}")
    print(f"{'-' * 40}\n")

print("=" * 70)
print("  ALL TRAINING COMPLETE")
print("=" * 70)
for name in MODEL_NAMES:
    r = all_results[name]
    print(f"  {name:15s} | best val acc: {r['best_val_acc']:.4f} (epoch {r['best_epoch']})")
    print(f"  {'':15s} | saved: best_{name}.pth")
print("=" * 70)

## 16) Training Curves (All Models)

In [ ]:
colors = {"resnet18": "#1f77b4", "unet": "#ff7f0e", "efficientnet": "#2ca02c"}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name in MODEL_NAMES:
    h = all_results[name]["history"]
    epochs = range(1, len(h["train_loss"]) + 1)
    c = colors[name]

    axes[0].plot(epochs, h["train_loss"], color=c, linestyle="--", alpha=0.5, label=f"{name} train")
    axes[0].plot(epochs, h["val_loss"], color=c, linestyle="-", label=f"{name} val")

    axes[1].plot(epochs, h["train_acc"], color=c, linestyle="--", alpha=0.5, label=f"{name} train")
    axes[1].plot(epochs, h["val_acc"], color=c, linestyle="-", label=f"{name} val")

for ax in axes:
    ax.axvline(x=PHASE1_EPOCHS, color="gray", linestyle=":", alpha=0.7, label="Phase boundary")
    ax.legend(fontsize=8)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss - All Models")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training & Validation Accuracy - All Models")

plt.tight_layout()
plt.show()
print("Training curves plotted for all 3 models.")

## 17) Evaluation - Per-Model Metrics

In [ ]:
results = {}

for name in MODEL_NAMES:
    print(f"\n{'=' * 60}")
    print(f"  EVALUATING: {name}")
    print(f"{'=' * 60}")

    model = create_model(name).to(device)
    ckpt_path = f"best_{name}.pth"
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"  Loaded: {ckpt_path}")

    criterion = nn.BCEWithLogitsLoss()
    test_loss, test_acc, all_probs, all_labels = eval_one_epoch(model, test_loader, criterion)

    all_preds = (all_probs >= 0.5).astype(float)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.0

    results[name] = {
        "acc": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "params": count_parameters(model),
        "all_probs": all_probs,
        "all_labels": all_labels,
        "all_preds": all_preds,
    }

    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")

    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Clean", "Corrupt"]).plot(ax=ax, cmap="Blues")
    ax.set_title(f"{name} - Confusion Matrix")
    plt.tight_layout()
    plt.show()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll 3 models evaluated on test set.")

## 18) Per-Corruption-Type Breakdown

In [ ]:
test_ctypes = []
test_labels_from_npz = []
test_npz_paths = sorted((OUT_DIR / "test").glob("*.npz"))
for p in test_npz_paths:
    d = np.load(p, allow_pickle=True)
    test_ctypes.append(str(d["ctype"]))
    test_labels_from_npz.append(int(d["label"]))

test_ctypes = np.array(test_ctypes)
unique_ctypes = sorted(set(test_ctypes))

print("=" * 80)
print("  PER-CORRUPTION-TYPE ACCURACY BREAKDOWN")
print("=" * 80)

header = f"{'Corruption Type':<30}"
for name in MODEL_NAMES:
    header += f" {name:>12}"
header += f" {'Count':>8}"
print(header)
print("-" * 80)

for ctype in unique_ctypes:
    mask = test_ctypes == ctype
    count = mask.sum()
    true_labels = np.array(test_labels_from_npz)[mask]
    row = f"{ctype:<30}"

    for name in MODEL_NAMES:
        preds = results[name]["all_preds"][mask]
        acc = (preds == true_labels).mean()
        row += f" {acc:>12.4f}"

    row += f" {count:>8}"
    print(row)

print("-" * 80)
print(f"{'OVERALL':<30}", end="")
for name in MODEL_NAMES:
    print(f" {results[name]['acc']:>12.4f}", end="")
print(f" {len(test_ctypes):>8}")
print("=" * 80)

## 19) Cross-Model Benchmarking Table

In [ ]:
print("=" * 70)
print("  CROSS-MODEL BENCHMARKING TABLE")
print("=" * 70)
print(f"{'Model':<20} {'Accuracy':>10} {'F1':>8} {'AUC-ROC':>10} {'Params':>12}")
print("-" * 65)
for name in MODEL_NAMES:
    r = results[name]
    print(f"{name:<20} {r['acc']:>10.4f} {r['f1']:>8.4f} {r['auc']:>10.4f} {r['params']:>12,}")
print("=" * 70)

## 20) Inference Time & Resource Measurement

In [ ]:
print("=" * 80)
print("  INFERENCE TIME & RESOURCE MEASUREMENT")
print("=" * 80)

dummy_single = torch.randn(1, 1, 64, 128, 128).to(device)
n_runs = 10

print(f"{'Model':<20} {'Inference (s)':>15} {'GPU Mem (GB)':>14} {'Parameters':>14}")
print("-" * 68)

for name in MODEL_NAMES:
    model = create_model(name).to(device)
    model.load_state_dict(torch.load(f"best_{name}.pth", map_location=device))
    model.eval()

    with torch.no_grad():
        _ = model(dummy_single)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t_start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy_single)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.time()

    avg_time = (t_end - t_start) / n_runs

    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_mem = 0.0

    params = count_parameters(model)
    print(f"{name:<20} {avg_time:>15.4f} {peak_mem:>14.2f} {params:>14,}")

    results[name]["inference_time"] = avg_time
    results[name]["gpu_memory_gb"] = peak_mem

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=" * 80)

## 21) Single-Scan Inference Demo

In [ ]:
demo_path = test_paths[0]
demo_vol = load_mhd(demo_path)
print(f"Demo scan: {demo_path}")
print(f"  Original shape: {demo_vol.shape}")

rng = np.random.default_rng(42)
test_cases = {}

test_cases["Clean"] = preprocess(demo_vol)

cropped, _ = apply_anatomical_cropping(demo_vol, removal_frac=0.30, mode="center", rng=np.random.default_rng(42))
test_cases["Anat. Cropping (center, 30%)"] = preprocess(cropped)

gaussian_blurred, _ = apply_gaussian_blur(demo_vol, sigma=2.5, rng=np.random.default_rng(42))
test_cases["Gaussian Blur (sigma=2.5)"] = preprocess(gaussian_blurred)

defocus_blurred, _ = apply_defocus_blur(demo_vol, sigma=2.0)
test_cases["Defocus Blur (sigma=2.0)"] = preprocess(defocus_blurred)

combo_params = {
    "anatomical_cropping": {"removal_frac": 0.25, "mode": "random"},
    "gaussian_blur": {"sigma": 2.0},
    "defocus_blur": {"sigma": 1.5}
}
combo_vol, _ = apply_combo_corruption(
    demo_vol,
    ["anatomical_cropping", "gaussian_blur", "defocus_blur"],
    combo_params,
    rng=np.random.default_rng(42)
)
test_cases["Combo_3 (all 3)"] = preprocess(combo_vol)

print(f"\n{'Test Case':<35}", end="")
for name in MODEL_NAMES:
    print(f" {name:>14}", end="")
print()
print("-" * 80)

for case_name, vol in test_cases.items():
    tensor = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).float().to(device)
    row = f"{case_name:<35}"
    for name in MODEL_NAMES:
        model = create_model(name).to(device)
        model.load_state_dict(torch.load(f"best_{name}.pth", map_location=device))
        model.eval()
        with torch.no_grad():
            logit = model(tensor)
            prob = torch.sigmoid(logit).item()
        row += f" {prob:>14.4f}"
        del model
    print(row)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("-" * 80)
print("Values shown: P(corrupted) - closer to 0 = clean, closer to 1 = corrupted")

## 22) Resource Summary

In [ ]:
total_time = time.time() - NOTEBOOK_START
hours = int(total_time // 3600)
mins = int((total_time % 3600) // 60)
secs = int(total_time % 60)

print("=" * 70)
print("  RESOURCE SUMMARY")
print("=" * 70)
print(f"  Total runtime:    {hours}h {mins}m {secs}s")
print(f"  Device:           {device}")
if torch.cuda.is_available():
    print(f"  GPU:              {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB total")
print(f"  Dataset size:     {len(train_ds)} train / {len(val_ds)} val / {len(test_ds)} test")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  Training epochs:  {PHASE1_EPOCHS} + {PHASE2_EPOCHS} per model")
print()

print("  Per-model summary:")
for name in MODEL_NAMES:
    r = results[name]
    print(f"    {name:15s} | acc={r['acc']:.4f} | F1={r['f1']:.4f} | AUC={r['auc']:.4f} "
          f"| params={r['params']:,} | infer={r['inference_time']:.3f}s | GPU={r['gpu_memory_gb']:.2f}GB")

print()
print("=" * 70)
print("  FINAL BENCHMARKING COMPLETE")
print("=" * 70)
print(f"  Models trained: {len(MODEL_NAMES)}")
print(f"  Corruption types: anatomical_cropping, gaussian_blur, defocus_blur + combo")
for name in MODEL_NAMES:
    r = results[name]
    print(f"  {name}: acc={r['acc']:.4f}, F1={r['f1']:.4f}, AUC={r['auc']:.4f}")
print("=" * 70)